<a href="https://colab.research.google.com/github/Avanthikai005/Terra-Incognito/blob/main/terra_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sat Sep 12 04:47:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os

PROJECT = "/content/drive/MyDrive/Terra_Incognita"

os.makedirs(PROJECT, exist_ok=True)
os.makedirs(PROJECT + "/models", exist_ok=True)
os.makedirs(PROJECT + "/results", exist_ok=True)
os.makedirs(PROJECT + "/notebooks", exist_ok=True)
os.makedirs(PROJECT + "/selected_data", exist_ok=True)

print(os.listdir(PROJECT))

['code', 'models', 'results', 'notebooks', 'selected_data']


In [9]:
!mkdir -p /content/xbd_project
%cd /content/xbd_project

!wget -c -O xbd_s12.tar.gz \
"https://zenodo.org/records/18960454/files/xbd_s12.tar.gz?download=1"

/content/xbd_project
--2026-09-12 04:54:02--  https://zenodo.org/records/18960454/files/xbd_s12.tar.gz?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 137.138.153.219, 188.184.103.118, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 206 PARTIAL_CONTENT
Length: 9523531926 (8.9G), 9500113354 (8.8G) remaining [application/octet-stream]
Saving to: ‘xbd_s12.tar.gz’

xbd_s12.tar.gz      100%[===================>]   8.87G  20.1MB/s    in 4m 38s  

2026-09-12 04:58:40 (32.6 MB/s) - ‘xbd_s12.tar.gz’ saved [9523531926/9523531926]



In [10]:
!ls -lh /content/xbd_project/

total 8.9G
-rw-r--r-- 1 root root 8.9G Sep 12 04:58 xbd_s12.tar.gz


In [11]:
!tar -xzf /content/xbd_project/xbd_s12.tar.gz \
-C /content/xbd_project/

In [12]:
!find /content/xbd_project -maxdepth 2 -type f | head -50

/content/xbd_project/xbd_s12.tar.gz
/content/xbd_project/xbd_s12/xbd_s12_metadata.geojson


In [13]:
!find /content/xbd_project -name "xbd_s12_metadata.geojson"

/content/xbd_project/xbd_s12/xbd_s12_metadata.geojson


In [20]:
METADATA_PATH = "/content/xbd_project/xbd_s12/xbd_s12_metadata.geojson"

In [21]:
!pip install -q geopandas

In [22]:
import geopandas as gpd
import pandas as pd

gdf = gpd.read_file(METADATA_PATH)
print("Number of patches:", len(gdf))
print(gdf.columns.tolist())

Number of patches: 10315
['xbd_uid', 'disaster', 'disaster_type', 'peril', 'xbd_tier', 'event_split', 'best_utm', 'N_intact', 'N_minor', 'N_major', 'N_destroyed', 'N_unclassified', 'N_total', 'xbd_date_pre', 'xbd_date_post', 'xbd_catalog_id_pre', 'xbd_catalog_id_post', 'xbd_sensor_pre', 'xbd_sensor_post', 'xbd_perc_nan_pre', 'xbd_perc_nan_post', 's2_mgrs_tile', 's2_date_pre', 's2_date_post', 's2_cs_pre', 's2_cs_post', 's1_date_pre', 's1_date_post', 's1_orbit_pre', 's1_orbit_post', 's1_direction_pre', 's1_direction_post', 's1_ids_pre', 's1_ids_post', 'geometry']


In [23]:
print(gdf[[
    "disaster",
    "disaster_type",
    "peril"
]].head(20))

print("Number of disasters:", gdf["disaster"].nunique())

print(gdf["disaster"].unique())

             disaster disaster_type    peril
0   guatemala-volcano       volcano  volcano
1   guatemala-volcano       volcano  volcano
2   guatemala-volcano       volcano  volcano
3   guatemala-volcano       volcano  volcano
4   guatemala-volcano       volcano  volcano
5   guatemala-volcano       volcano  volcano
6   guatemala-volcano       volcano  volcano
7   guatemala-volcano       volcano  volcano
8   guatemala-volcano       volcano  volcano
9   guatemala-volcano       volcano  volcano
10  guatemala-volcano       volcano  volcano
11  guatemala-volcano       volcano  volcano
12  guatemala-volcano       volcano  volcano
13  guatemala-volcano       volcano  volcano
14  guatemala-volcano       volcano  volcano
15  guatemala-volcano       volcano  volcano
16  guatemala-volcano       volcano  volcano
17  guatemala-volcano       volcano  volcano
18  guatemala-volcano       volcano  volcano
19  guatemala-volcano       volcano  volcano
Number of disasters: 16
['guatemala-volcano' 'hurricane

In [24]:
event_table = (
    gdf.groupby(["disaster", "disaster_type", "peril"])
       .size()
       .reset_index(name="patches")
       .sort_values("patches", ascending=False)
)

display(event_table)

,disaster,disaster_type,peril,patches
11,portugal-wildfire,fire,wildfire,1869
10,pinery-bushfire,fire,wildfire,1845
13,socal-fire,fire,wildfire,1403
15,woolsey-fire,fire,wildfire,878
8,nepal-flooding,flooding,flood,619
4,hurricane-michael,wind,storm,550
1,hurricane-florence,flooding,flood,546
2,hurricane-harvey,flooding,flood,522
7,midwest-flooding,flooding,flood,445
3,hurricane-matthew,wind,storm,405


In [25]:
summary = (
    gdf.groupby("disaster")
       .agg(
           patches=("xbd_uid", "count"),
           disaster_type=("disaster_type", "first"),
           peril=("peril", "first"),
           buildings=("N_total", "sum"),
           destroyed=("N_destroyed", "sum"),
           major=("N_major", "sum"),
           minor=("N_minor", "sum"),
           intact=("N_intact", "sum")
       )
       .sort_values("patches", ascending=False)
)

summary["damage_rate"] = (
    (summary["destroyed"] +
     summary["major"] +
     summary["minor"])
    / summary["buildings"]
)

display(summary)

,patches,disaster_type,peril,buildings,destroyed,major,minor,intact,damage_rate
disaster,,,,,,,,,
portugal-wildfire,1869,fire,wildfire,23413,1090,296,176,20787,0.066715
pinery-bushfire,1845,fire,wildfire,5961,229,99,82,5027,0.068780
socal-fire,1403,fire,wildfire,18969,2333,110,136,15697,0.135959
woolsey-fire,878,fire,wildfire,7015,1876,126,189,4638,0.312331
nepal-flooding,619,flooding,flood,43265,502,4721,5134,31225,0.239385
hurricane-michael,550,wind,storm,35501,1225,2919,8292,22692,0.350300
hurricane-florence,546,flooding,flood,11548,81,1949,232,8466,0.195878
hurricane-harvey,522,flooding,flood,37955,848,13378,4510,18638,0.493637
midwest-flooding,445,flooding,flood,13896,165,193,246,12819,0.043466


In [28]:
# Show every disaster event and its disaster type
event_table = (
    gdf.groupby(["disaster", "disaster_type", "peril"])
       .agg(
           patches=("xbd_uid", "count"),
           buildings=("N_total", "sum"),
           intact=("N_intact", "sum"),
           minor=("N_minor", "sum"),
           major=("N_major", "sum"),
           destroyed=("N_destroyed", "sum")
       )
       .reset_index()
       .sort_values("patches", ascending=False)
)

display(event_table)

,disaster,disaster_type,peril,patches,buildings,intact,minor,major,destroyed
11,portugal-wildfire,fire,wildfire,1869,23413,20787,176,296,1090
10,pinery-bushfire,fire,wildfire,1845,5961,5027,82,99,229
13,socal-fire,fire,wildfire,1403,18969,15697,136,110,2333
15,woolsey-fire,fire,wildfire,878,7015,4638,189,126,1876
8,nepal-flooding,flooding,flood,619,43265,31225,5134,4721,502
4,hurricane-michael,wind,storm,550,35501,22692,8292,2919,1225
1,hurricane-florence,flooding,flood,546,11548,8466,232,1949,81
2,hurricane-harvey,flooding,flood,522,37955,18638,4510,13378,848
7,midwest-flooding,flooding,flood,445,13896,12819,246,193,165
3,hurricane-matthew,wind,storm,405,23964,4058,12331,2717,3524
